In [ ]:
# ==================== SETUP & CONFIG ====================
import os
from google.colab import drive

drive.mount('/content/drive')

SYNAPSE = "/content/drive/MyDrive/synapse"

# ---- Paths ----
CHECKPOINT_DIR  = os.path.join(SYNAPSE, "checkpoints")
TOKENIZER_PATH  = os.path.join(SYNAPSE, "tokenizer_out", "tokenizer.json")
SHARD_DIR       = os.path.join(SYNAPSE, "token_shards_merged")
META_PATH       = os.path.join(SHARD_DIR, "meta.json")
MANIFEST_PATH   = os.path.join(SYNAPSE, "manifests", "training_latest.json")

# ---- Checkpoint selector ----
CHECKPOINT_NAME = "synapse_2b_d2560_l28.pth"
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAME)
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Tokenizer:  {TOKENIZER_PATH}")
print(f"Shards:     {SHARD_DIR}")

# Verify paths exist
for label, path in [("Checkpoint", CHECKPOINT_PATH), ("Tokenizer", TOKENIZER_PATH), ("Shard dir", SHARD_DIR)]:
    exists = os.path.exists(path)
    print(f"  {label}: {'OK' if exists else 'MISSING'}")
    if not exists:
        print(f"    -> {path}")

In [ ]:
# ==================== MODEL DEFINITION ====================
import json
import math
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tokenizers import Tokenizer

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---- Architecture constants ----
BLOCK_SIZE    = 2048
EMBED_DIM     = 2560
NUM_LAYERS    = 28
NUM_HEADS     = 20
NUM_KV_HEADS  = 4
FF_HIDDEN_DIM = 6912
ROPE_BASE     = 10000.0
RMSNORM_EPS   = 1e-5

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        in_dtype = x.dtype
        x = x.float()
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x * rms).to(in_dtype) * self.weight

def precompute_rope(head_dim, max_seq_len, base, device, dtype=torch.float32):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device, dtype=dtype) / head_dim))
    t = torch.arange(max_seq_len, device=device, dtype=dtype)
    freqs = torch.outer(t, inv_freq)
    cos = torch.cat([freqs.cos(), freqs.cos()], dim=-1)
    sin = torch.cat([freqs.sin(), freqs.sin()], dim=-1)
    return cos, sin

def apply_rope(x, cos, sin):
    T = x.size(-2)
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    half = x.size(-1) // 2
    x1, x2 = x[..., :half], x[..., half:]
    rotated = torch.cat([-x2, x1], dim=-1)
    return (x * cos) + (rotated * sin)

class CausalGQA(nn.Module):
    def __init__(self, embed_dim, num_heads, num_kv_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        assert num_heads % num_kv_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = embed_dim // num_heads
        self.n_rep = num_heads // num_kv_heads
        kv_dim = num_kv_heads * self.head_dim
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, kv_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, kv_dim, bias=False)
        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.num_heads,    self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2, bias=False)
        self.w2 = nn.Linear(hidden_dim, embed_dim, bias=False)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, num_kv_heads, ff_hidden_dim, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps=rms_eps)
        self.attn  = CausalGQA(embed_dim, num_heads, num_kv_heads)
        self.norm2 = RMSNorm(embed_dim, eps=rms_eps)
        self.ff    = SwiGLU(embed_dim, ff_hidden_dim)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x

class SynapseGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.blocks = nn.ModuleList([
            TransformerBlock(EMBED_DIM, NUM_HEADS, NUM_KV_HEADS, FF_HIDDEN_DIM, RMSNORM_EPS)
            for _ in range(NUM_LAYERS)
        ])
        self.final_norm = RMSNorm(EMBED_DIM, eps=RMSNORM_EPS)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight  # tied
        head_dim = EMBED_DIM // NUM_HEADS
        cos, sin = precompute_rope(head_dim, BLOCK_SIZE, ROPE_BASE, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward(self, idx):
        x = self.token_embedding(idx)
        cos, sin = self.rope_cos, self.rope_sin
        for block in self.blocks:
            x = block(x, cos, sin)
        return self.lm_head(self.final_norm(x))

# ==================== LOAD MODEL ====================
with open(META_PATH, "r") as f:
    meta = json.load(f)
VOCAB_SIZE = int(meta["vocab_size"])
if VOCAB_SIZE % 64 != 0:
    VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64
SHARD_DTYPE = np.dtype(meta.get("shard_dtype", "uint16"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load checkpoint to CPU first (v2 checkpoints are ~24 GB: model + optimizer + metadata).
# We only need model weights (~8 GB) for evaluation.
file_size = os.path.getsize(CHECKPOINT_PATH) / 1024 / 1024
print(f"Checkpoint: {CHECKPOINT_PATH} ({file_size:.1f} MB)")
print("Loading to CPU (to avoid GPU OOM on optimizer state)...")

ckpt_obj = torch.load(CHECKPOINT_PATH, map_location="cpu")
if isinstance(ckpt_obj, dict) and ckpt_obj.get("schema") == "v2":
    state_dict = ckpt_obj["model"]
    ckpt_step = ckpt_obj.get("curr_step", "?")
    ckpt_seen = len(ckpt_obj.get("seen_shards", []))
    print(f"v2 checkpoint: step={ckpt_step}, seen_shards={ckpt_seen}")
    # Free optimizer state and other large fields immediately
    del ckpt_obj["optimizer"]
    del ckpt_obj
    gc.collect()
else:
    state_dict = ckpt_obj
    ckpt_step = "?"
    ckpt_seen = 0
    print("legacy checkpoint (model-only)")
    del ckpt_obj
    gc.collect()

# Clean _orig_mod. prefix from torch.compile
clean_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
del state_dict
gc.collect()

# Build model on CPU, load weights, then move to GPU
print("Building model and loading weights...")
model = SynapseGPT(VOCAB_SIZE)
model_state = model.state_dict()
matched = {k: v for k, v in clean_state.items() if k in model_state and v.shape == model_state[k].shape}
model.load_state_dict(matched, strict=False)
print(f"Loaded {len(matched)}/{len(model_state)} layers")

del clean_state, matched
gc.collect()

# Move to GPU
model = model.to(device)
model.eval()

# Load tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
EOT_ID = tokenizer.token_to_id("<|endoftext|>")

n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel on {device} | Vocab: {VOCAB_SIZE} | Params: {n_params/1e9:.3f}B | EOT: {EOT_ID}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ==================== GENERATE FUNCTION ====================

@torch.no_grad()
def generate(prompt, max_tokens=300, temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.2):
    """Autoregressive generation. Returns (full_text, continuation_only)."""
    ids = tokenizer.encode(prompt).ids
    prompt_len = len(ids)
    tokens = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_tokens):
        context = tokens[:, -BLOCK_SIZE:]
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(context)[:, -1, :]

        # Windowed repetition penalty (last 256 tokens)
        if repetition_penalty > 1.0:
            recent = set(tokens[0, -256:].tolist())
            for tid in recent:
                if tid < logits.size(-1):
                    if logits[0, tid] > 0:
                        logits[0, tid] /= repetition_penalty
                    else:
                        logits[0, tid] *= repetition_penalty

        logits = logits / temperature

        # Top-k
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        # Top-p (nucleus)
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = float('-inf')
            logits = torch.zeros_like(logits).scatter(1, sorted_indices, sorted_logits)

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        if next_token.item() == EOT_ID:
            break

        tokens = torch.cat([tokens, next_token], dim=1)

    output_ids = tokens[0].tolist()
    full_text = tokenizer.decode(output_ids)
    continuation = tokenizer.decode(output_ids[prompt_len:])
    return full_text, continuation

# Quick smoke test
_, sample = generate("The meaning of life is", max_tokens=50)
print(f"Smoke test: {sample[:200]}")

In [ ]:
# ==================== TEST 1: PERPLEXITY ON SHARDS ====================
# Measures how well the model predicts text from merged training shards.
# Lower = better. Rough guide: <50 decent, <20 good, <10 very good.
import random

EVAL_CHUNKS = 50
CHUNK_LENGTH = BLOCK_SIZE  # 2048

@torch.no_grad()
def compute_perplexity(shard_dir, num_chunks, chunk_len):
    manifest_path = os.path.join(shard_dir, "shard_manifest.json")
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    shards = manifest["shards"]
    if not shards:
        print("No shards in manifest!")
        return None

    # Pick a random shard (deterministic)
    pick_rng = random.Random(42)
    shard_info = pick_rng.choice(shards)
    shard_path = os.path.join(shard_dir, shard_info["shard"])
    print(f"Evaluating on shard: {shard_info['shard']}")

    tokens = np.fromfile(shard_path, dtype=SHARD_DTYPE).astype(np.int64)
    total_tokens = len(tokens)
    print(f"Loaded {total_tokens:,} tokens")

    if total_tokens < chunk_len + 1:
        print("Shard too small to evaluate!")
        return None

    np_rng = np.random.default_rng(42)
    starts = np_rng.integers(0, total_tokens - chunk_len - 1, size=num_chunks)

    losses = []
    for s in starts:
        chunk = tokens[s : s + chunk_len + 1]
        x = torch.tensor(chunk[:-1][None, :], dtype=torch.long, device=device)
        y = torch.tensor(chunk[1:][None, :], dtype=torch.long, device=device)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        losses.append(loss.item())

    avg_loss = sum(losses) / len(losses)
    ppl = math.exp(avg_loss)

    print(f"\nPerplexity Results ({num_chunks} chunks of {chunk_len} tokens):")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Perplexity:   {ppl:.2f}")
    print(f"  Min Loss:     {min(losses):.4f} (best chunk)")
    print(f"  Max Loss:     {max(losses):.4f} (worst chunk)")
    return {"avg_loss": avg_loss, "perplexity": ppl, "min_loss": min(losses), "max_loss": max(losses)}

ppl_result = compute_perplexity(SHARD_DIR, EVAL_CHUNKS, CHUNK_LENGTH)

In [ ]:
# ==================== TEST 2: REPETITION RATE ====================
# Measures how often the model repeats itself. Lower = better.
from collections import Counter

def repetition_rate(text, n):
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    counts = Counter(ngrams)
    repeated = sum(c - 1 for c in counts.values() if c > 1)
    return repeated / len(ngrams) if ngrams else 0.0

test_prompts = [
    "Once upon a time",
    "The knight drew his sword and",
    "The best way to learn programming is",
    "Scientists recently discovered that",
    "According to a new study published in",
    "I think the most important thing about",
    "Here are three reasons why",
]

print("Repetition Rate Analysis")
print("=" * 70)
print(f"{'Prompt':<40} {'2-gram':>8} {'3-gram':>8} {'4-gram':>8}")
print("-" * 70)

all_outputs = []
for prompt in test_prompts:
    output, _ = generate(prompt, max_tokens=300, temperature=0.8, top_k=40)
    all_outputs.append(output)
    r2 = repetition_rate(output, 2)
    r3 = repetition_rate(output, 3)
    r4 = repetition_rate(output, 4)
    label = prompt[:37] + "..." if len(prompt) > 37 else prompt
    print(f"{label:<40} {r2:>7.1%} {r3:>7.1%} {r4:>7.1%}")

combined = " ".join(all_outputs)
print("-" * 70)
print(f"{'OVERALL':<40} {repetition_rate(combined, 2):>7.1%} {repetition_rate(combined, 3):>7.1%} {repetition_rate(combined, 4):>7.1%}")
print("\nGuide: <10% is good, 10-25% is okay, >25% is concerning")

In [ ]:
# ==================== TEST 3: TOKEN DIVERSITY ====================
# Unique tokens / total tokens. Higher = better.
# Low diversity suggests mode collapse.

def token_diversity(text):
    ids = tokenizer.encode(text).ids
    total = len(ids)
    unique = len(set(ids))
    return unique, total, unique / total if total > 0 else 0.0

print("Token Diversity Analysis")
print("=" * 70)
print(f"{'Prompt':<40} {'Unique':>7} {'Total':>7} {'Ratio':>8}")
print("-" * 70)

total_unique_all = set()
total_count_all = 0

for prompt, output in zip(test_prompts, all_outputs):
    unique, total, ratio = token_diversity(output)
    ids = tokenizer.encode(output).ids
    total_unique_all.update(ids)
    total_count_all += total
    label = prompt[:37] + "..." if len(prompt) > 37 else prompt
    print(f"{label:<40} {unique:>7} {total:>7} {ratio:>7.1%}")

print("-" * 70)
overall_ratio = len(total_unique_all) / total_count_all if total_count_all > 0 else 0
print(f"{'OVERALL':<40} {len(total_unique_all):>7} {total_count_all:>7} {overall_ratio:>7.1%}")
print(f"\nVocab utilization: {len(total_unique_all)} / {VOCAB_SIZE} tokens used ({len(total_unique_all)/VOCAB_SIZE:.1%})")
print("Guide: >30% diversity is good, <15% is concerning")

In [ ]:
# ==================== TEST 4: COHERENCE CHECK ====================
# Multiple continuations from same prompt.
# Good model: varied but thematically consistent.

coherence_prompts = [
    "The knight drew his sword and",
    "Deep in the mountains there lived",
    "The key to a successful business is",
    "New research suggests that climate change",
]

NUM_SAMPLES = 3

for prompt in coherence_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    for i in range(NUM_SAMPLES):
        _, continuation = generate(prompt, max_tokens=150, temperature=0.9, top_k=50)
        print(f"\n--- Sample {i+1} ---")
        print(continuation.strip()[:500])

In [ ]:
# ==================== TEST 5: GOLD RESPONSE PERPLEXITY ====================
# Perplexity on known-good responses, conditioned on the prompt.
# Measures generalization — can the model predict correct answers?
# Lower = better. Track across checkpoints to see improvement.

@torch.no_grad()
def compute_response_perplexity(prompt_text, response_text):
    """Perplexity only on response tokens (conditioned on prompt)."""
    full_text = prompt_text + response_text
    full_ids = tokenizer.encode(full_text).ids
    prompt_ids = tokenizer.encode(prompt_text).ids
    prefix_len = len(prompt_ids)

    if len(full_ids) < prefix_len + 1:
        return float('inf')
    if len(full_ids) > BLOCK_SIZE + 1:
        full_ids = full_ids[:BLOCK_SIZE + 1]

    x = torch.tensor([full_ids[:-1]], dtype=torch.long, device=device)
    y = torch.tensor([full_ids[1:]], dtype=torch.long, device=device)

    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        logits = model(x)

    # Only score response tokens
    logits_response = logits[:, prefix_len - 1:, :]
    targets_response = y[:, prefix_len - 1:]

    if targets_response.numel() == 0:
        return float('inf')

    loss = F.cross_entropy(logits_response.reshape(-1, VOCAB_SIZE), targets_response.reshape(-1))
    return math.exp(loss.item())

# Curated prompts with gold responses (knowledge + reasoning)
GOLD_PROMPTS = [
    # Knowledge
    {"cat": "knowledge", "prompt": "The capital of Australia is", "gold": " Canberra, not Sydney or Melbourne as many people assume."},
    {"cat": "knowledge", "prompt": "Water is composed of", "gold": " two hydrogen atoms and one oxygen atom, giving it the chemical formula H2O."},
    {"cat": "knowledge", "prompt": "The speed of light in a vacuum is approximately", "gold": " 299,792 kilometers per second, or about 186,000 miles per second."},
    {"cat": "knowledge", "prompt": "Shakespeare wrote Romeo and Juliet", "gold": " in approximately 1594-1596, during the early part of his career as a playwright."},
    {"cat": "knowledge", "prompt": "The first moon landing occurred in", "gold": " 1969, when Apollo 11 astronauts Neil Armstrong and Buzz Aldrin walked on the lunar surface."},
    {"cat": "knowledge", "prompt": "Photosynthesis is the process by which", "gold": " plants convert sunlight, water, and carbon dioxide into glucose and oxygen."},
    {"cat": "knowledge", "prompt": "The largest ocean on Earth is the", "gold": " Pacific Ocean, covering more than 63 million square miles."},
    # Reasoning
    {"cat": "reasoning", "prompt": "If a shirt costs $25 and is on sale for 20% off, the final price is", "gold": " $20, because the discount is $25 times 0.20 which equals $5."},
    {"cat": "reasoning", "prompt": "The next number in the sequence 2, 6, 18, 54 is", "gold": " 162, because each number is multiplied by 3 to get the next one."},
    {"cat": "reasoning", "prompt": "If all dogs are animals and some animals are pets, then", "gold": " we can conclude that all dogs are animals, but we cannot conclude that all dogs are pets."},
    {"cat": "reasoning", "prompt": "A farmer has 12 apples, gives away 3, then buys twice as many as he has. He now has", "gold": " 27 apples: 12 minus 3 is 9, then 9 times 2 is 18, plus the 9 he kept equals 27."},
    {"cat": "reasoning", "prompt": "Sorting the numbers 42, 7, 91, 3, 28 from smallest to largest gives", "gold": " 3, 7, 28, 42, 91."},
    # Language understanding
    {"cat": "language", "prompt": "The opposite of hot is", "gold": " cold, just as the opposite of tall is short."},
    {"cat": "language", "prompt": "A synonym for happy is", "gold": " joyful, glad, or content."},
    {"cat": "language", "prompt": "The plural of mouse is", "gold": " mice, which is an irregular plural form in English."},
]

print("Gold Response Perplexity (lower = better)")
print("=" * 70)
print(f"{'Prompt':<50} {'Category':<12} {'PPL':>8}")
print("-" * 70)

ppl_by_cat = {}
for item in GOLD_PROMPTS:
    ppl = compute_response_perplexity(item["prompt"], item["gold"])
    cat = item["cat"]
    ppl_by_cat.setdefault(cat, []).append(ppl)
    label = item["prompt"][:47] + "..." if len(item["prompt"]) > 47 else item["prompt"]
    print(f"{label:<50} {cat:<12} {ppl:>8.1f}")

print("-" * 70)
print("\nMedian PPL by category:")
all_ppls = []
for cat, ppls in sorted(ppl_by_cat.items()):
    median_ppl = sorted(ppls)[len(ppls) // 2]
    all_ppls.extend(ppls)
    print(f"  {cat:<12}: {median_ppl:.1f}")
overall_median = sorted(all_ppls)[len(all_ppls) // 2]
print(f"  {'OVERALL':<12}: {overall_median:.1f}")

In [ ]:
# ==================== TEST 6: SEMANTIC SIMILARITY ====================
# Tests whether the model's embeddings encode semantic relationships.
# Close-meaning pairs should have higher cosine similarity than far pairs.
# The gap between groups should widen as training progresses.
import matplotlib.pyplot as plt

# Compute global embedding mean once (the frequency-direction proxy)
# baked in by tied input/output weights — subtracting it leaves the
# secondary structure that actually reflects semantics.
with torch.no_grad():
    EMB_MEAN = model.token_embedding.weight.mean(dim=0)

def get_word_embedding(word):
    """Mean-centered embedding. Uses a leading space so BPE picks the
    in-context token variant rather than a rare word-initial piece."""
    ids = tokenizer.encode(" " + word).ids
    with torch.no_grad():
        embs = model.token_embedding.weight[ids] - EMB_MEAN  # (num_subwords, EMBED_DIM)
    return embs.mean(dim=0)  # average if multi-token

def cosine_sim(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# Word pairs: semantically close vs. semantically far
CLOSE_PAIRS = [
    ("king", "queen"),
    ("dog", "cat"),
    ("hot", "warm"),
    ("run", "sprint"),
    ("happy", "joy"),
    ("car", "vehicle"),
    ("doctor", "nurse"),
    ("ocean", "sea"),
    ("big", "large"),
    ("fast", "quick"),
]

FAR_PAIRS = [
    ("king", "banana"),
    ("dog", "algebra"),
    ("hot", "democracy"),
    ("run", "furniture"),
    ("happy", "volcano"),
    ("car", "photosynthesis"),
    ("doctor", "skateboard"),
    ("ocean", "pencil"),
    ("big", "saxophone"),
    ("fast", "philosophy"),
]

print("Semantic Similarity Analysis (Embedding Cosine Similarity)")
print("=" * 70)

close_sims = []
far_sims = []

print(f"\n{'CLOSE PAIRS (should be HIGH)':}")
print(f"{'Pair':<30} {'Cosine Sim':>10}")
print("-" * 42)
for w1, w2 in CLOSE_PAIRS:
    e1 = get_word_embedding(w1)
    e2 = get_word_embedding(w2)
    sim = cosine_sim(e1, e2)
    close_sims.append(sim)
    print(f"{w1 + ' / ' + w2:<30} {sim:>10.4f}")

print(f"\n{'FAR PAIRS (should be LOW)':}")
print(f"{'Pair':<30} {'Cosine Sim':>10}")
print("-" * 42)
for w1, w2 in FAR_PAIRS:
    e1 = get_word_embedding(w1)
    e2 = get_word_embedding(w2)
    sim = cosine_sim(e1, e2)
    far_sims.append(sim)
    print(f"{w1 + ' / ' + w2:<30} {sim:>10.4f}")

avg_close = sum(close_sims) / len(close_sims)
avg_far = sum(far_sims) / len(far_sims)
gap = avg_close - avg_far

print(f"\n{'='*42}")
print(f"Average CLOSE similarity: {avg_close:.4f}")
print(f"Average FAR similarity:   {avg_far:.4f}")
print(f"GAP (close - far):        {gap:.4f}")
print(f"\nInterpretation:")
if gap > 0.15:
    print(f"  GOOD: Clear semantic structure (gap={gap:.3f})")
elif gap > 0.05:
    print(f"  EMERGING: Some semantic structure forming (gap={gap:.3f})")
elif gap > 0:
    print(f"  WEAK: Minimal semantic differentiation (gap={gap:.3f})")
else:
    print(f"  NONE: No semantic structure yet (gap={gap:.3f}) — early training expected")

# Bar chart visualization
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(CLOSE_PAIRS))
width = 0.35

bars1 = ax.bar(x - width/2, close_sims, width, label=f'Close pairs (avg={avg_close:.3f})', color='#2ecc71', alpha=0.8)
bars2 = ax.bar(x + width/2, far_sims, width, label=f'Far pairs (avg={avg_far:.3f})', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Word Pair Index')
ax.set_ylabel('Cosine Similarity')
ax.set_title(f'Embedding Semantic Similarity (Gap = {gap:.4f})')
ax.set_xticks(x)
pair_labels = [f"{c[0]}/{c[1]}" for c in CLOSE_PAIRS]
ax.set_xticklabels(pair_labels, rotation=45, ha='right', fontsize=8)
ax.legend()
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== TEST 7: WEIGHT DISTRIBUTION ====================
# Visualize weight statistics to check for instability or dead layers.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("SynapseGPT Weight Analysis", fontsize=14, fontweight="bold")

# --- Panel 1: Weight magnitude by layer type ---
ax = axes[0, 0]
layer_types = {
    "q_proj": [], "k_proj": [], "v_proj": [], "o_proj": [],
    "ff.w1": [], "ff.w2": [],
}
for name, param in model.named_parameters():
    for lt in layer_types:
        if lt in name and "weight" in name:
            layer_types[lt].append(param.detach().cpu().float().abs().mean().item())

positions = range(len(layer_types))
means = [np.mean(v) if v else 0 for v in layer_types.values()]
stds = [np.std(v) if v else 0 for v in layer_types.values()]
ax.bar(positions, means, yerr=stds, capsize=4, color='steelblue', alpha=0.8)
ax.set_xticks(positions)
ax.set_xticklabels(layer_types.keys(), rotation=30)
ax.set_ylabel("Mean |weight|")
ax.set_title("Mean Weight Magnitude by Layer Type")

# --- Panel 2: RMSNorm scale values ---
ax = axes[0, 1]
norm_scales = []
norm_labels = []
for name, param in model.named_parameters():
    if "norm" in name and "weight" in name:
        vals = param.detach().cpu().float().numpy()
        norm_scales.append(vals)
        norm_labels.append(name.split(".")[1])  # block index

if norm_scales:
    norm_means = [v.mean() for v in norm_scales]
    ax.scatter(range(len(norm_means)), norm_means, s=20, alpha=0.7, color='darkorange')
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='init=1.0')
    ax.set_xlabel("RMSNorm layer index")
    ax.set_ylabel("Mean scale")
    ax.set_title("RMSNorm Scale Values")
    ax.legend()

# --- Panel 3: Weight histogram (all params) ---
ax = axes[1, 0]
all_weights = []
for name, param in model.named_parameters():
    if param.dim() >= 2:  # skip norms
        all_weights.append(param.detach().cpu().float().flatten().numpy())
if all_weights:
    combined = np.concatenate(all_weights)
    # Sample to avoid OOM on plot
    if len(combined) > 1_000_000:
        rng = np.random.default_rng(42)
        combined = rng.choice(combined, 1_000_000, replace=False)
    ax.hist(combined, bins=200, density=True, color='purple', alpha=0.7)
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel("Weight value")
    ax.set_ylabel("Density")
    ax.set_title(f"Weight Distribution (sampled 1M params)")
    ax.set_xlim(-0.1, 0.1)

# --- Panel 4: Per-layer weight std ---
ax = axes[1, 1]
block_stds = []
for i in range(NUM_LAYERS):
    block_params = []
    for name, param in model.named_parameters():
        if f"blocks.{i}." in name and param.dim() >= 2:
            block_params.append(param.detach().cpu().float().std().item())
    if block_params:
        block_stds.append(np.mean(block_params))
if block_stds:
    ax.plot(range(len(block_stds)), block_stds, 'o-', color='teal', markersize=4)
    ax.set_xlabel("Block index")
    ax.set_ylabel("Mean weight std")
    ax.set_title("Per-Block Weight Standard Deviation")

plt.tight_layout()
plt.show()

In [ ]:
# ==================== TEST 8: LLM-AS-JUDGE (DeepSeek) ====================
# Score generated responses on 4 criteria using DeepSeek.
# OPTIONAL: skip this cell if no API key available.
import time
from openai import OpenAI

# Load API key from Colab Secrets or environment
try:
    from google.colab import userdata
    DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
except Exception:
    DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    print("No DEEPSEEK_API_KEY found. Skipping LLM-as-judge.")
    print("To enable: add DEEPSEEK_API_KEY to Colab Secrets or .env")
    judge_results = None
else:
    judge_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

    JUDGE_SYSTEM = """You are an expert evaluator of AI-generated text. Score the response on each criterion (1-5):

1. **Coherence** (1-5): Grammatically correct, fluent, logically structured?
2. **Relevance** (1-5): Does it address what was asked?
3. **Accuracy** (1-5): Factually correct? (For creative tasks, rate appropriateness)
4. **Instruction_Following** (1-5): Follows format/constraints requested?

Respond ONLY with valid JSON:
{"coherence": N, "relevance": N, "accuracy": N, "instruction_following": N, "note": "one sentence"}"""

    JUDGE_PROMPTS = [
        "List exactly three benefits of exercise.",
        "Explain what a neural network is in simple terms.",
        "If a shirt costs $25 and is 20% off, what is the final price?",
        "What is the capital of Australia?",
        "Write a short poem about the ocean.",
        "What are some tips for a beginner learning to cook?",
        "Sort these numbers from smallest to largest: 42, 7, 91, 3, 28.",
        "Explain why reading books is beneficial.",
        "Name three countries in South America.",
        "What comes next in the sequence: 2, 6, 18, 54, ...?",
        "The best way to learn a new language is",
        "Describe the water cycle in two sentences.",
        "What makes a good leader?",
        "Give advice for someone starting a new job.",
        "Explain gravity to a five-year-old.",
    ]

    def judge_response(prompt, response, max_retries=3):
        user_msg = f"PROMPT:\n{prompt}\n\nRESPONSE:\n{response}"
        for attempt in range(max_retries):
            try:
                completion = judge_client.chat.completions.create(
                    model="deepseek-chat",
                    messages=[
                        {"role": "system", "content": JUDGE_SYSTEM},
                        {"role": "user", "content": user_msg},
                    ],
                    temperature=0.0,
                    max_tokens=200,
                )
                raw = completion.choices[0].message.content.strip()
                if raw.startswith("```"):
                    raw = raw.split("```")[1]
                    if raw.startswith("json"):
                        raw = raw[4:]
                scores = json.loads(raw)
                for key in ["coherence", "relevance", "accuracy", "instruction_following"]:
                    assert key in scores and 1 <= scores[key] <= 5
                return scores
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(2)
                else:
                    return {"coherence": 0, "relevance": 0, "accuracy": 0,
                            "instruction_following": 0, "note": f"ERROR: {e}"}

    print(f"Judging {len(JUDGE_PROMPTS)} responses with DeepSeek...")
    judge_results = []

    for i, prompt in enumerate(JUDGE_PROMPTS):
        _, response = generate(prompt, max_tokens=200, temperature=0.7, top_k=40)
        scores = judge_response(prompt, response)
        scores["prompt"] = prompt
        scores["response"] = response[:300]
        judge_results.append(scores)
        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(JUDGE_PROMPTS)} done")
        time.sleep(0.5)

    # Summary
    score_cols = ["coherence", "relevance", "accuracy", "instruction_following"]
    print(f"\n{'='*60}")
    print("LLM-AS-JUDGE RESULTS")
    print(f"{'='*60}")
    print(f"{'Criterion':<25} {'Mean':>6} {'Min':>5} {'Max':>5}")
    print("-" * 45)
    for col in score_cols:
        vals = [r[col] for r in judge_results if r[col] > 0]
        if vals:
            print(f"{col:<25} {sum(vals)/len(vals):>6.2f} {min(vals):>5} {max(vals):>5}")
    avg_all = np.mean([[r[c] for c in score_cols] for r in judge_results if r["coherence"] > 0])
    print(f"-" * 45)
    print(f"{'OVERALL AVERAGE':<25} {avg_all:>6.2f}")

In [ ]:
# ==================== SUMMARY DASHBOARD ====================
# Collects all metrics for easy comparison across checkpoints.

print("\n" + "#" * 60)
print("# SYNAPSE EVALUATION SUMMARY")
print("#" * 60)
print(f"\nCheckpoint: {CHECKPOINT_NAME}")
print(f"Step: {ckpt_step}")
print(f"Seen shards: {ckpt_seen}")

summary = {"checkpoint": CHECKPOINT_NAME, "step": ckpt_step, "seen_shards": ckpt_seen}

# Perplexity
if ppl_result:
    print(f"\n--- Perplexity (on training shards) ---")
    print(f"  Loss: {ppl_result['avg_loss']:.4f} | PPL: {ppl_result['perplexity']:.2f}")
    summary["shard_loss"] = ppl_result["avg_loss"]
    summary["shard_ppl"] = ppl_result["perplexity"]

# Repetition
overall_rep = repetition_rate(combined, 3)
print(f"\n--- Repetition (3-gram, overall) ---")
print(f"  {overall_rep:.1%}")
summary["rep_3gram"] = overall_rep

# Diversity
print(f"\n--- Token Diversity (overall) ---")
print(f"  {overall_ratio:.1%}")
summary["token_diversity"] = overall_ratio

# Gold PPL
if all_ppls:
    print(f"\n--- Gold Response Perplexity (median) ---")
    print(f"  {overall_median:.1f}")
    summary["gold_ppl_median"] = overall_median

# Semantic gap
print(f"\n--- Semantic Similarity Gap ---")
print(f"  Close avg: {avg_close:.4f} | Far avg: {avg_far:.4f} | Gap: {gap:.4f}")
summary["semantic_gap"] = gap

# Judge scores
if judge_results:
    print(f"\n--- LLM Judge (avg score, 1-5) ---")
    print(f"  {avg_all:.2f}")
    summary["judge_avg"] = float(avg_all)

# Save summary
summary_path = os.path.join(SYNAPSE, "eval_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved: {summary_path}")
print("\nRun this notebook at multiple checkpoints and compare eval_summary.json to track evolution.")